In [2]:
# ============================================================
# FLAGSHIP PROJECT — ADVANCED ANALYSIS
#
# 1. ML Explainability
# 2. Rolling Performance
# 3. Portfolio Allocation Analysis
# ============================================================

import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")


# ============================================================
# PATHS
# ============================================================

DATA_DIR = "data/raw"

BACKTEST_DIR = (
    "data/processed/final_backtest"
)

EVALUATION_DIR = (
    "data/processed/final_evaluation"
)

OUTPUT_DIR = (
    "outputs/advanced_analysis"
)

FIGURE_DIR = (
    "outputs/figures"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

os.makedirs(
    FIGURE_DIR,
    exist_ok=True
)


# ============================================================
# CONFIGURATION
# ============================================================

TRADING_DAYS = 252

ROLLING_WINDOW = 252

HORIZON = 21

TRAINING_WINDOW = 504

RANDOM_STATE = 42


# ============================================================
# FEATURES
# ============================================================

FEATURES = [

    "return_5d",

    "return_21d",

    "return_63d",

    "return_126d",

    "volatility_21d",

    "volatility_63d",

    "price_vs_ma21",

    "price_vs_ma63",

    "price_vs_ma126",

    "RSI_14",

    "drawdown_126d",

    "skewness_63d",

    "kurtosis_63d"
]


# ============================================================
# LOAD DATA
# ============================================================

prices = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "adjusted_close_prices.csv"
    ),
    index_col="Date",
    parse_dates=True
)

returns = pd.read_csv(
    os.path.join(
        DATA_DIR,
        "daily_returns.csv"
    ),
    index_col="Date",
    parse_dates=True
)

strategy_returns = pd.read_csv(
    os.path.join(
        BACKTEST_DIR,
        "all_strategy_returns.csv"
    ),
    index_col="Date",
    parse_dates=True
)

cumulative_returns = pd.read_csv(
    os.path.join(
        EVALUATION_DIR,
        "cumulative_performance.csv"
    ),
    index_col="Date",
    parse_dates=True
)

evaluation_returns = pd.read_csv(
    os.path.join(
        EVALUATION_DIR,
        "evaluation_returns.csv"
    ),
    index_col="Date",
    parse_dates=True
)


prices = prices.sort_index()
returns = returns.sort_index()
strategy_returns = strategy_returns.sort_index()
evaluation_returns = evaluation_returns.sort_index()


# ============================================================
# UNIVERSE
# ============================================================

minimum_observations = int(
    len(prices) * 0.90
)

stocks = [

    stock

    for stock in prices.columns

    if prices[stock].count()
    >= minimum_observations
]

prices = prices[stocks]

returns = returns[stocks]


print("=" * 75)
print("ADVANCED ANALYSIS")
print("=" * 75)

print(
    f"\nStocks: {len(stocks)}"
)


# ============================================================
# FEATURE ENGINEERING
# ============================================================

def create_features(
    price,
    daily_return
):

    df = pd.DataFrame(
        index=price.index
    )

    df["return_5d"] = (
        price.pct_change(5)
    )

    df["return_21d"] = (
        price.pct_change(21)
    )

    df["return_63d"] = (
        price.pct_change(63)
    )

    df["return_126d"] = (
        price.pct_change(126)
    )

    df["volatility_21d"] = (
        daily_return
        .rolling(21)
        .std()
        * np.sqrt(252)
    )

    df["volatility_63d"] = (
        daily_return
        .rolling(63)
        .std()
        * np.sqrt(252)
    )

    ma_21 = (
        price
        .rolling(21)
        .mean()
    )

    ma_63 = (
        price
        .rolling(63)
        .mean()
    )

    ma_126 = (
        price
        .rolling(126)
        .mean()
    )

    df["price_vs_ma21"] = (
        price / ma_21 - 1
    )

    df["price_vs_ma63"] = (
        price / ma_63 - 1
    )

    df["price_vs_ma126"] = (
        price / ma_126 - 1
    )

    delta = price.diff()

    gains = delta.clip(
        lower=0
    )

    losses = -delta.clip(
        upper=0
    )

    avg_gain = (
        gains
        .rolling(14)
        .mean()
    )

    avg_loss = (
        losses
        .rolling(14)
        .mean()
    )

    rs = (
        avg_gain
        / avg_loss.replace(
            0,
            np.nan
        )
    )

    df["RSI_14"] = (
        100
        - (
            100
            / (1 + rs)
        )
    )

    rolling_high = (
        price
        .rolling(126)
        .max()
    )

    df["drawdown_126d"] = (
        price
        / rolling_high
        - 1
    )

    df["skewness_63d"] = (
        daily_return
        .rolling(63)
        .skew()
    )

    df["kurtosis_63d"] = (
        daily_return
        .rolling(63)
        .kurt()
    )

    return df


# ============================================================
# BUILD PANEL DATA
# ============================================================

feature_data = {}

records = []


for stock in stocks:

    feature_df = create_features(
        prices[stock],
        returns[stock]
    )

    target = (
        prices[stock]
        .shift(-HORIZON)
        / prices[stock]
        - 1
    )

    temp = feature_df.copy()

    temp["target"] = target

    temp["stock"] = stock

    temp = temp.dropna()

    feature_data[
        stock
    ] = feature_df

    records.append(
        temp
    )


ml_data = pd.concat(
    records
).sort_index()


# ============================================================
# ============================================================
# 1. ML EXPLAINABILITY
# ============================================================
# ============================================================

print("\n" + "=" * 75)
print("1. ML EXPLAINABILITY")
print("=" * 75)


# ------------------------------------------------------------
# Model definitions
# ------------------------------------------------------------

def create_random_forest():

    return RandomForestRegressor(

        n_estimators=300,

        max_depth=6,

        min_samples_leaf=20,

        max_features="sqrt",

        random_state=RANDOM_STATE,

        n_jobs=-1
    )


def create_xgboost():

    return XGBRegressor(

        n_estimators=300,

        max_depth=3,

        learning_rate=0.03,

        subsample=0.8,

        colsample_bytree=0.8,

        min_child_weight=10,

        reg_alpha=0.1,

        reg_lambda=1.0,

        objective="reg:squarederror",

        random_state=RANDOM_STATE,

        n_jobs=-1
    )


models = {

    "Random_Forest":
        create_random_forest,

    "XGBoost":
        create_xgboost
}


# ------------------------------------------------------------
# Use the same walk-forward dates as the main backtest
# ------------------------------------------------------------

dates = prices.index

start_position = max(
    TRAINING_WINDOW,
    126 + HORIZON
)

rebalance_positions = list(
    range(
        start_position,
        len(dates) - HORIZON,
        21
    )
)


importance_records = []


# ------------------------------------------------------------
# Train models repeatedly and collect importance
# ------------------------------------------------------------

for model_name, factory in models.items():

    print(
        f"\nCollecting "
        f"{model_name} feature importance..."
    )

    for position in rebalance_positions:

        current_date = (
            dates[position]
        )

        train_start = dates[
            position
            - TRAINING_WINDOW
        ]

        train_end = current_date

        train_data = ml_data[
            (
                ml_data.index
                >= train_start
            )
            &
            (
                ml_data.index
                < train_end
            )
        ]

        if len(train_data) < 300:

            continue

        X_train = (
            train_data[
                FEATURES
            ]
        )

        y_train = (
            train_data[
                "target"
            ]
        )

        model = factory()

        model.fit(
            X_train,
            y_train
        )

        importances = (
            model
            .feature_importances_
        )

        for feature, importance in zip(
            FEATURES,
            importances
        ):

            importance_records.append({

                "Date":
                    current_date,

                "Model":
                    model_name,

                "Feature":
                    feature,

                "Importance":
                    importance
            })


importance_df = pd.DataFrame(
    importance_records
)


# ------------------------------------------------------------
# Average feature importance
# ------------------------------------------------------------

average_importance = (
    importance_df
    .groupby(
        ["Model", "Feature"]
    )["Importance"]
    .mean()
    .reset_index()
)


average_importance.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "ml_feature_importance.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# Normalize within each model
# ------------------------------------------------------------

normalized_importance = (
    average_importance
    .copy()
)

normalized_importance[
    "Normalized_Importance"
] = (
    normalized_importance
    .groupby("Model")[
        "Importance"
    ]
    .transform(
        lambda x:
        x / x.sum()
    )
)


normalized_importance.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "ml_feature_importance_normalized.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# Feature importance chart
# ------------------------------------------------------------

for model_name in models.keys():

    temp = (
        normalized_importance[
            normalized_importance[
                "Model"
            ] == model_name
        ]
        .sort_values(
            "Normalized_Importance"
        )
        .tail(10)
    )

    plt.figure(
        figsize=(10, 7)
    )

    plt.barh(
        temp["Feature"],
        temp["Normalized_Importance"]
    )

    plt.title(
        f"{model_name}: Top 10 Feature Importance",
        fontsize=15,
        fontweight="bold"
    )

    plt.xlabel(
        "Normalized Importance"
    )

    plt.ylabel(
        "Feature"
    )

    plt.grid(
        axis="x",
        alpha=0.25
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            FIGURE_DIR,
            f"{model_name}_feature_importance.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


# ------------------------------------------------------------
# Compare RF vs XGBoost
# ------------------------------------------------------------

comparison = (
    normalized_importance
    .pivot(
        index="Feature",
        columns="Model",
        values="Normalized_Importance"
    )
    .fillna(0)
)

comparison = (
    comparison
    .sort_values(
        "XGBoost",
        ascending=False
    )
)


comparison.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "rf_vs_xgboost_features.csv"
    )
)


plt.figure(
    figsize=(12, 8)
)

comparison.plot(
    kind="barh",
    figsize=(12, 8)
)

plt.title(
    "Random Forest vs XGBoost Feature Importance",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel(
    "Normalized Importance"
)

plt.ylabel(
    "Feature"
)

plt.grid(
    axis="x",
    alpha=0.25
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "RF_vs_XGBoost_feature_importance.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ============================================================
# ============================================================
# 2. ROLLING PERFORMANCE
# ============================================================
# ============================================================

print("\n" + "=" * 75)
print("2. ROLLING PERFORMANCE")
print("=" * 75)


# ------------------------------------------------------------
# Rolling Sharpe
# ------------------------------------------------------------

rolling_sharpe = pd.DataFrame(
    index=strategy_returns.index
)


for strategy in (
    strategy_returns.columns
):

    rolling_mean = (
        strategy_returns[
            strategy
        ]
        .rolling(
            ROLLING_WINDOW
        )
        .mean()
        * TRADING_DAYS
    )

    rolling_std = (
        strategy_returns[
            strategy
        ]
        .rolling(
            ROLLING_WINDOW
        )
        .std()
        * np.sqrt(
            TRADING_DAYS
        )
    )

    rolling_sharpe[
        strategy
    ] = (
        rolling_mean
        / rolling_std
    )


rolling_sharpe.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "rolling_12m_sharpe.csv"
    )
)


plt.figure(
    figsize=(14, 7)
)

for strategy in (
    rolling_sharpe.columns
):

    plt.plot(
        rolling_sharpe.index,
        rolling_sharpe[strategy],
        label=strategy,
        linewidth=1.5
    )

plt.axhline(
    0,
    linewidth=0.8
)

plt.title(
    "12-Month Rolling Sharpe Ratio",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel(
    "Date"
)

plt.ylabel(
    "Rolling Sharpe"
)

plt.legend(
    frameon=False
)

plt.grid(
    alpha=0.25
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "05_rolling_sharpe.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ------------------------------------------------------------
# Rolling Volatility
# ------------------------------------------------------------

rolling_volatility = pd.DataFrame(
    index=strategy_returns.index
)


for strategy in (
    strategy_returns.columns
):

    rolling_volatility[
        strategy
    ] = (
        strategy_returns[
            strategy
        ]
        .rolling(
            ROLLING_WINDOW
        )
        .std()
        * np.sqrt(
            TRADING_DAYS
        )
    )


rolling_volatility.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "rolling_12m_volatility.csv"
    )
)


plt.figure(
    figsize=(14, 7)
)

for strategy in (
    rolling_volatility.columns
):

    plt.plot(
        rolling_volatility.index,
        rolling_volatility[strategy] * 100,
        label=strategy,
        linewidth=1.5
    )

plt.title(
    "12-Month Rolling Volatility",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel(
    "Date"
)

plt.ylabel(
    "Annualized Volatility (%)"
)

plt.legend(
    frameon=False
)

plt.grid(
    alpha=0.25
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "06_rolling_volatility.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ------------------------------------------------------------
# Rolling Alpha vs NIFTY
# ------------------------------------------------------------

if "NIFTY_50" in evaluation_returns.columns:

    nifty = (
        evaluation_returns[
            "NIFTY_50"
        ]
    )

    rolling_alpha = pd.DataFrame(
        index=evaluation_returns.index
    )

    for strategy in (
        strategy_returns.columns
    ):

        if strategy not in (
            evaluation_returns.columns
        ):

            continue

        portfolio = (
            evaluation_returns[
                strategy
            ]
        )

        covariance = (
            portfolio
            .rolling(
                ROLLING_WINDOW
            )
            .cov(
                nifty
            )
        )

        nifty_variance = (
            nifty
            .rolling(
                ROLLING_WINDOW
            )
            .var()
        )

        rolling_beta = (
            covariance
            / nifty_variance
        )

        rolling_alpha[
            strategy
        ] = (

            (
                portfolio
                .rolling(
                    ROLLING_WINDOW
                )
                .mean()
            )

            -

            (
                rolling_beta
                * nifty
                .rolling(
                    ROLLING_WINDOW
                )
                .mean()
            )
        ) * TRADING_DAYS


    rolling_alpha.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "rolling_alpha_vs_nifty.csv"
        )
    )


    plt.figure(
        figsize=(14, 7)
    )

    for strategy in (
        rolling_alpha.columns
    ):

        plt.plot(
            rolling_alpha.index,
            rolling_alpha[strategy] * 100,
            label=strategy,
            linewidth=1.5
        )

    plt.axhline(
        0,
        linewidth=0.8
    )

    plt.title(
        "12-Month Rolling Alpha vs NIFTY 50",
        fontsize=15,
        fontweight="bold"
    )

    plt.xlabel(
        "Date"
    )

    plt.ylabel(
        "Annualized Alpha (%)"
    )

    plt.legend(
        frameon=False
    )

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            FIGURE_DIR,
            "07_rolling_alpha_vs_nifty.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


# ============================================================
# ============================================================
# 3. PORTFOLIO ALLOCATION ANALYSIS
# ============================================================
# ============================================================

print("\n" + "=" * 75)
print("3. PORTFOLIO ALLOCATION ANALYSIS")
print("=" * 75)


# ============================================================
# LOAD WEIGHTS
# ============================================================

weight_files = {

    "Equal_Weight":
        "Equal_Weight_weights.csv",

    "Minimum_Variance":
        "Minimum_Variance_weights.csv",

    "Maximum_Sharpe":
        "Maximum_Sharpe_weights.csv",

    "Risk_Parity":
        "Risk_Parity_weights.csv",

    "Random_Forest":
        "Random_Forest_weights.csv",

    "XGBoost":
        "XGBoost_weights.csv"
}


weight_data = {}


for strategy, filename in (
    weight_files.items()
):

    path = os.path.join(
        BACKTEST_DIR,
        filename
    )

    if not os.path.exists(path):

        print(
            f"Missing: {filename}"
        )

        continue

    df = pd.read_csv(
        path
    )

    if "Date" in df.columns:

        df["Date"] = pd.to_datetime(
            df["Date"]
        )

        df = df.set_index(
            "Date"
        )

    weight_data[
        strategy
    ] = df


# ============================================================
# AVERAGE ALLOCATION
# ============================================================

average_allocations = {}


for strategy, df in (
    weight_data.items()
):

    numeric_df = (
        df
        .select_dtypes(
            include=np.number
        )
    )

    average_allocations[
        strategy
    ] = (
        numeric_df
        .mean()
        .sort_values(
            ascending=False
        )
    )


average_allocation_df = pd.DataFrame(
    average_allocations
)


average_allocation_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "average_stock_allocation.csv"
    )
)


# ============================================================
# TOP STOCK ALLOCATIONS
# ============================================================

top_allocation_records = []


for strategy, series in (
    average_allocations.items()
):

    top_stocks = (
        series
        .head(10)
    )

    for rank, (
        stock,
        weight
    ) in enumerate(
        top_stocks.items(),
        start=1
    ):

        top_allocation_records.append({

            "Strategy":
                strategy,

            "Rank":
                rank,

            "Stock":
                stock,

            "Average_Weight":
                weight
        })


top_allocations = pd.DataFrame(
    top_allocation_records
)


top_allocations.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "top_10_average_allocations.csv"
    ),
    index=False
)


# ============================================================
# CONCENTRATION METRICS
# ============================================================

concentration_records = []


for strategy, df in (
    weight_data.items()
):

    numeric_df = (
        df
        .select_dtypes(
            include=np.number
        )
    )

    # HHI:
    # Sum of squared portfolio weights

    hhi = (
        numeric_df ** 2
    ).sum(axis=1)

    # Effective number of stocks
    effective_n = (
        1 / hhi
    )

    # Largest position
    max_weight = (
        numeric_df.max(axis=1)
    )

    # Number of meaningful positions
    number_positions = (
        numeric_df.gt(0.01)
        .sum(axis=1)
    )

    concentration_records.append({

        "Strategy":
            strategy,

        "Average_HHI":
            hhi.mean(),

        "Average_Effective_Stocks":
            effective_n.mean(),

        "Average_Max_Weight":
            max_weight.mean(),

        "Maximum_Max_Weight":
            max_weight.max(),

        "Average_Positions_Above_1pct":
            number_positions.mean()
    })


concentration_df = pd.DataFrame(
    concentration_records
).set_index(
    "Strategy"
)


concentration_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "portfolio_concentration.csv"
    )
)


# ============================================================
# TOP 10 ALLOCATION CHART
# ============================================================

# Focus on ML strategies for the flagship story.

for strategy in [
    "Random_Forest",
    "XGBoost",
    "Maximum_Sharpe"
]:

    if strategy not in average_allocations:

        continue

    top = (
        average_allocations[
            strategy
        ]
        .head(10)
        .sort_values()
    )

    plt.figure(
        figsize=(10, 7)
    )

    plt.barh(
        top.index,
        top.values * 100
    )

    plt.title(
        f"{strategy}: Top 10 Average Portfolio Weights",
        fontsize=15,
        fontweight="bold"
    )

    plt.xlabel(
        "Average Weight (%)"
    )

    plt.ylabel(
        "Stock"
    )

    plt.grid(
        axis="x",
        alpha=0.25
    )

    plt.tight_layout()

    safe_name = (
        strategy
        .lower()
    )

    plt.savefig(
        os.path.join(
            FIGURE_DIR,
            f"08_{safe_name}_allocation.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


# ============================================================
# WEIGHT EVOLUTION
# ============================================================

for strategy in [
    "XGBoost",
    "Random_Forest",
    "Maximum_Sharpe"
]:

    if strategy not in weight_data:

        continue

    df = weight_data[
        strategy
    ].copy()

    # Average weights across time
    avg = (
        df.mean()
        .sort_values(
            ascending=False
        )
    )

    # Select top 8 stocks
    top_stocks = (
        avg
        .head(8)
        .index
    )

    plt.figure(
        figsize=(14, 7)
    )

    for stock in top_stocks:

        if stock not in df.columns:

            continue

        plt.plot(
            df.index,
            df[stock] * 100,
            label=stock,
            linewidth=1.4
        )

    plt.title(
        f"{strategy}: Portfolio Weight Evolution",
        fontsize=15,
        fontweight="bold"
    )

    plt.xlabel(
        "Rebalance Date"
    )

    plt.ylabel(
        "Portfolio Weight (%)"
    )

    plt.legend(
        frameon=False
    )

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()

    safe_name = (
        strategy
        .lower()
    )

    plt.savefig(
        os.path.join(
            FIGURE_DIR,
            f"09_{safe_name}_weight_evolution.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("ADVANCED ANALYSIS COMPLETE")
print("=" * 75)

print("\nFiles created:")

for root, dirs, files in os.walk(
    OUTPUT_DIR
):

    for file in sorted(files):

        print(
            os.path.join(
                root,
                file
            )
        )


print("\nFigures created:")

for file in sorted(
    os.listdir(
        FIGURE_DIR
    )
):

    if file.startswith(
        (
            "RF_",
            "XGBoost_",
            "05_",
            "06_",
            "07_",
            "08_",
            "09_"
        )
    ):

        print(
            os.path.join(
                FIGURE_DIR,
                file
            )
        )

ADVANCED ANALYSIS

Stocks: 15

1. ML EXPLAINABILITY



2. ROLLING PERFORMANCE

3. PORTFOLIO ALLOCATION ANALYSIS

ADVANCED ANALYSIS COMPLETE

Files created:
outputs/advanced_analysis/average_stock_allocation.csv
outputs/advanced_analysis/ml_feature_importance.csv
outputs/advanced_analysis/ml_feature_importance_normalized.csv
outputs/advanced_analysis/portfolio_concentration.csv
outputs/advanced_analysis/rf_vs_xgboost_features.csv
outputs/advanced_analysis/rolling_12m_sharpe.csv
outputs/advanced_analysis/rolling_12m_volatility.csv
outputs/advanced_analysis/rolling_alpha_vs_nifty.csv
outputs/advanced_analysis/top_10_average_allocations.csv

Figures created:
outputs/figures/05_rolling_sharpe.png
outputs/figures/06_rolling_volatility.png
outputs/figures/07_rolling_alpha_vs_nifty.png
outputs/figures/08_maximum_sharpe_allocation.png
outputs/figures/08_random_forest_allocation.png
outputs/figures/08_xgboost_allocation.png
outputs/figures/09_maximum_sharpe_weight_evolution.png
outputs/figures/0

<Figure size 1200x800 with 0 Axes>